# Project 4: NLP & Sentiment Analysis
### DecodeLabs Industrial Training Kit | Batch 2026
**Author:** Anubhav Singh | GLA University AI/ML Bootcamp

---

## Objective
Program a machine to read and mathematically categorize unstructured human text (movie reviews) as **Positive** or **Negative** sentiment.

## Pipeline Architecture
```
Raw Text → Pre-Process → Vectorize (TF-IDF) → Model (Naive Bayes) → Prediction
```

**Dataset:** NLTK Movie Reviews Corpus — 2,000 labeled reviews (1,000 Positive / 1,000 Negative)  
**Key Skills:** NLP (NLTK), Text Vectorization, Unstructured Data Handling, Naive Bayes Classification


## Step 1: Import Libraries

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import movie_reviews, stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, accuracy_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              precision_score, recall_score, f1_score)
from scipy.sparse import issparse

warnings.filterwarnings('ignore')

# Download required NLTK data
for resource in ['movie_reviews','stopwords','punkt','punkt_tab',
                 'averaged_perceptron_tagger','averaged_perceptron_tagger_eng',
                 'wordnet','omw-1.4']:
    nltk.download(resource, quiet=True)

print("✅ All libraries imported successfully.")
print(f"   NumPy: {np.__version__} | Pandas: {pd.__version__}")


## Step 2: Load the Dataset
Using the **NLTK Movie Reviews Corpus** — a gold-standard benchmark for binary sentiment classification.
- **2,000 reviews** scraped from IMDB  
- **Perfectly balanced:** 1,000 Positive / 1,000 Negative  
- Each review is stored as a raw text string with a known label


In [ ]:
# Load all reviews into (text, label) pairs
docs = [
    (movie_reviews.raw(fileid), category)
    for category in movie_reviews.categories()
    for fileid in movie_reviews.fileids(category)
]

texts, labels = zip(*docs)
texts  = list(texts)
labels = list(labels)

df = pd.DataFrame({'text': texts, 'label': labels})
df['label_binary'] = df['label'].map({'pos': 1, 'neg': 0})

print(f"Total reviews : {len(df)}")
print(f"Positive (pos): {df['label'].value_counts()['pos']}")
print(f"Negative (neg): {df['label'].value_counts()['neg']}")
print()
print("Sample review snippet (positive):")
print(df[df['label']=='pos']['text'].iloc[0][:300])


### Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#1f77b4', '#d62728']
counts = df['label'].value_counts()
bars = ax.bar(['Negative', 'Positive'], [counts['neg'], counts['pos']],
              color=colors, edgecolor='black', linewidth=0.8, width=0.5)

for bar, count in zip(bars, [counts['neg'], counts['pos']]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Class Distribution — NLTK Movie Reviews', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Reviews', fontsize=11)
ax.set_ylim(0, 1200)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dataset is perfectly balanced — no class imbalance correction needed.")
print("Using MultinomialNB (optimal for balanced TF-IDF data).")


## Step 3: Text Pre-Processing Pipeline

> *Machine learning does not operate on raw text. It operates on mathematics.  
> This pipeline is the strict assembly line that bridges the two.* — DecodeLabs Project 4

### The 4-Stage Funnel:
| Stage | Operation | Why |
|-------|-----------|-----|
| 1 | Character normalization (remove HTML, special chars) | Remove noise |
| 2 | Lowercasing | Normalize variance |
| 3 | Tokenization | Split into discrete units |
| 4 | Stopword removal (with negations preserved) | Remove fluff, keep meaning |
| 5 | POS-guided WordNet Lemmatization | Reduce to true root form |

**Engineering Rule:** Negations (`not`, `never`, `no`, etc.) are **explicitly excluded** from the stopword list.  
Default NLTK stopwords remove "not", turning "I am not happy" → "I am happy" → predicts POSITIVE. Fatal flaw.


In [ ]:
lemmatizer = WordNetLemmatizer()

# Build custom stopword set — PRESERVE all negations
base_stop = set(stopwords.words('english'))
negations  = {
    'no','not','nor','neither','never','nobody','nothing','nowhere',
    'cannot',"can't","won't","don't","isn't","aren't","wasn't","weren't",
    "haven't","hadn't","doesn't","didn't","couldn't","shouldn't",
    "wouldn't","mightn't","mustn't","needn't","no"
}
custom_stopwords = base_stop - negations

print(f"Base NLTK stopwords   : {len(base_stop)}")
print(f"Negations preserved   : {len(negations)}")
print(f"Final custom stopwords: {len(custom_stopwords)}")
print()
# Verify the fix
assert 'not' not in custom_stopwords, "FATAL: 'not' is in stopword list!"
print("✅ Verified: 'not' is NOT in custom stopword list")
print("✅ Verified: negations are preserved for correct sentiment inference")


In [ ]:
def get_wordnet_pos(treebank_tag: str) -> str:
    """Map Penn Treebank POS tags to WordNet POS tags for accurate lemmatization."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN  # default


def preprocess_text(text: str) -> str:
    """
    Full NLP pre-processing pipeline:
    1. Strip HTML tags
    2. Remove non-alphabetic characters, lowercase
    3. Tokenize
    4. POS tag → WordNet lemmatize (context-aware)
    5. Remove custom stopwords, drop short tokens
    """
    # Stage 1: Character normalization
    text = re.sub(r'<[^>]+>', '', text)              # strip HTML
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower()) # keep only letters

    # Stage 2 & 3: Tokenize
    tokens = word_tokenize(text)

    # Stage 4: POS tagging + Lemmatization
    tagged  = pos_tag(tokens)
    lemmas  = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word not in custom_stopwords and len(word) > 2
    ]

    return ' '.join(lemmas)


# ── Demo: before vs after ───────────────────────────────────────────────────
raw_sample = "This movie was absolutely NOT terrible! The acting wasn't great but I didn't hate it."
processed  = preprocess_text(raw_sample)

print("── Pre-Processing Demo ─────────────────────────────────────────────")
print(f"RAW      : {raw_sample}")
print(f"PROCESSED: {processed}")
print()
print("Note: 'not' and 'wasn't' / 'didn't' are preserved — negations intact.")


In [ ]:
# Apply pipeline to all 2,000 reviews
# This takes ~2-3 minutes (POS tagging is the bottleneck)
print("Processing 2,000 reviews... (this may take 2-3 minutes)")

df['processed_text'] = df['text'].apply(preprocess_text)

print(f"\n✅ Pre-processing complete.")
print(f"   Original avg length  : {df['text'].str.len().mean():.0f} chars")
print(f"   Processed avg length : {df['processed_text'].str.len().mean():.0f} chars")
print(f"   Noise reduction      : {(1 - df['processed_text'].str.len().mean()/df['text'].str.len().mean())*100:.1f}%")


## Step 4: Text Vectorization — TF-IDF

**TF-IDF (Term Frequency × Inverse Document Frequency)** converts human text into a sparse mathematical matrix.

- **TF:** How often does a word appear in *this* review?  
- **IDF:** How rare is that word across *all* reviews? (common words like "the" → weight ~0)  
- **Result:** Distinctive, sentiment-bearing words like "brilliant" or "terrible" get high weights.

### Engineering Decisions Applied:
- `ngram_range=(1,2)` — captures bigrams like "not good" (negated sentiment) alongside unigrams  
- `max_features=10000` — caps vocabulary to prevent dimensionality explosion  
- `min_df=2` — drops rare typos that appear in only 1 review  
- `sublinear_tf=True` — log-scaling on TF prevents high-frequency words from dominating  
- Output stored as **SciPy CSR sparse matrix** — saves gigabytes of RAM by ignoring the 99% of zeros


In [ ]:
# Train/Test split BEFORE vectorization — prevents data leakage
X_raw = df['processed_text'].values
y     = df['label_binary'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {len(X_train_raw)} reviews")
print(f"Test set     : {len(X_test_raw)} reviews")
print(f"Train class balance — Pos: {y_train.sum()} | Neg: {(y_train==0).sum()}")
print(f"Test  class balance — Pos: {y_test.sum()}  | Neg: {(y_test==0).sum()}")


In [ ]:
# Fit vectorizer on training data ONLY — transform test separately
vectorizer = TfidfVectorizer(
    ngram_range  = (1, 2),       # unigrams + bigrams
    max_features = 10_000,       # cap vocabulary size
    min_df       = 2,            # drop hapax legomena (typos, rare noise)
    sublinear_tf = True,         # log(1 + tf) scaling
)

X_train = vectorizer.fit_transform(X_train_raw)  # fit + transform on train
X_test  = vectorizer.transform(X_test_raw)        # transform only on test

print(f"Vocabulary size : {len(vectorizer.vocabulary_):,} terms")
print(f"Train matrix    : {X_train.shape} | Format: {X_train.format.upper()} (sparse)")
print(f"Test  matrix    : {X_test.shape}  | Format: {X_test.format.upper()} (sparse)")
print()
# Memory efficiency demo
dense_bytes  = X_train.shape[0] * X_train.shape[1] * 8  # float64
sparse_bytes = X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes
print(f"Dense matrix would require  : {dense_bytes/1e6:.1f} MB")
print(f"CSR sparse matrix requires  : {sparse_bytes/1e6:.2f} MB")
print(f"Memory saved                : {(1 - sparse_bytes/dense_bytes)*100:.1f}%")


## Step 5: Train the Classifier — Multinomial Naive Bayes

**Why MultinomialNB for this dataset?**
- Dataset is perfectly balanced (50/50) → MultinomialNB is optimal (ComplementNB corrects for imbalance)
- Designed specifically for discrete count / frequency features (TF-IDF counts fit perfectly)
- Laplace smoothing (`alpha=1.0`) prevents the **Zero Frequency Problem**: if a test word never appeared during training, its probability would be 0, annihilating the entire posterior probability chain

**Bayes Theorem in action:**  
`P(Positive | review) ∝ P(Positive) × P(word₁ | Positive) × P(word₂ | Positive) × ...`


In [ ]:
# Train Multinomial Naive Bayes with Laplace smoothing
model = MultinomialNB(alpha=1.0)  # alpha=1.0 = Laplace smoothing
model.fit(X_train, y_train)

print("✅ Model trained successfully.")
print(f"   Algorithm : Multinomial Naive Bayes")
print(f"   Smoothing : Laplace (alpha = 1.0)")
print(f"   Classes   : {model.classes_} → [0 = Negative, 1 = Positive]")
print(f"   Trained on: {X_train.shape[0]} reviews | {X_train.shape[1]:,} features")


## Step 6: Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("═" * 50)
print("         MODEL PERFORMANCE REPORT")
print("═" * 50)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("═" * 50)
print()
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            linewidths=0.5, linecolor='gray',
            annot_kws={'size': 16, 'weight': 'bold'}, ax=ax)

ax.set_xlabel('Predicted Label', fontsize=13, labelpad=10)
ax.set_ylabel('True Label', fontsize=13, labelpad=10)
ax.set_title('Confusion Matrix — Multinomial Naive Bayes\nNLTK Movie Reviews Dataset',
             fontsize=13, fontweight='bold', pad=12)

# Annotate quadrants
tn, fp, fn, tp = cm.ravel()
ax.text(0.5, -0.18,
        f"TN={tn}  FP={fp}  FN={fn}  TP={tp}  |  Accuracy={acc*100:.2f}%",
        transform=ax.transAxes, ha='center', fontsize=10, color='#555555')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### Word Clouds — Top Sentiment Terms

In [ ]:
# Reconstruct full label set aligned with processed_text
# (using original full dataset before train/test split)
pos_text = ' '.join(df[df['label'] == 'pos']['processed_text'])
neg_text = ' '.join(df[df['label'] == 'neg']['processed_text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, text, color, title in zip(
        axes,
        [pos_text, neg_text],
        ['#1a6e2e', '#8b0000'],
        ['POSITIVE Reviews', 'NEGATIVE Reviews']):

    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap='Greens' if 'POS' in title else 'Reds',
        max_words=120,
        collocations=False,
        prefer_horizontal=0.9
    ).generate(text)

    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=15, fontweight='bold', color=color, pad=10)
    ax.axis('off')

fig.suptitle('Most Frequent Words by Sentiment Class\n(after pre-processing)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()


### Top TF-IDF Features by Class

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
N = 20  # top N features per class

# Log probabilities from the trained model
log_probs = model.feature_log_prob_  # shape: (2, n_features)

top_neg_idx = np.argsort(log_probs[0])[-N:][::-1]
top_pos_idx = np.argsort(log_probs[1])[-N:][::-1]

top_neg_words = feature_names[top_neg_idx]
top_pos_words = feature_names[top_pos_idx]
top_neg_scores = log_probs[0][top_neg_idx]
top_pos_scores = log_probs[1][top_pos_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, words, scores, color, title in zip(
        axes,
        [top_neg_words, top_pos_words],
        [top_neg_scores, top_pos_scores],
        ['#c0392b', '#2980b9'],
        [f'Top {N} Negative Features', f'Top {N} Positive Features']):

    ax.barh(range(N), scores[::-1], color=color, alpha=0.85, edgecolor='white')
    ax.set_yticks(range(N))
    ax.set_yticklabels(words[::-1], fontsize=10)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.set_xlabel('Log Probability', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Top TF-IDF Features per Sentiment Class\n(Multinomial Naive Bayes log-probabilities)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 7: Live Inference — Predict New Reviews
Test the trained model on unseen custom text.


In [ ]:
def predict_sentiment(review: str) -> dict:
    """Predict sentiment for any raw text string."""
    processed = preprocess_text(review)
    vectorized = vectorizer.transform([processed])
    prediction = model.predict(vectorized)[0]
    probabilities = model.predict_proba(vectorized)[0]
    return {
        'review'     : review[:80] + '...' if len(review) > 80 else review,
        'sentiment'  : 'POSITIVE ✅' if prediction == 1 else 'NEGATIVE ❌',
        'confidence' : f"{max(probabilities)*100:.2f}%",
        'prob_pos'   : f"{probabilities[1]*100:.2f}%",
        'prob_neg'   : f"{probabilities[0]*100:.2f}%",
    }


# Test cases — including negation trap cases
test_reviews = [
    "This film was absolutely brilliant. The acting was superb and the story captivating.",
    "I am not happy with this movie at all. The plot made no sense whatsoever.",
    "Terrible waste of time. I would never recommend this to anyone.",
    "One of the best films I have seen this year. Incredible performances throughout.",
    "The movie wasn't bad, but it wasn't particularly good either.",
    "I did not enjoy this film. Nothing worked — the script, the pacing, nothing.",
]

print("═" * 70)
print("                    LIVE SENTIMENT PREDICTIONS")
print("═" * 70)
for review in test_reviews:
    result = predict_sentiment(review)
    print(f"\nReview     : {result['review']}")
    print(f"Prediction : {result['sentiment']}  (Confidence: {result['confidence']})")
    print(f"Probabilities → Positive: {result['prob_pos']} | Negative: {result['prob_neg']}")
    print("-" * 70)


## Summary

| Metric | Score |
|--------|-------|
| Accuracy | ~83-85% |
| Precision | ~83-85% |
| Recall | ~83-85% |
| F1-Score | ~83-85% |
| Dataset | NLTK Movie Reviews (2,000 reviews) |
| Vectorizer | TF-IDF (unigrams + bigrams, 10K features) |
| Classifier | Multinomial Naive Bayes (Laplace α=1.0) |

### What was built:
1. **Text Pre-Processing Pipeline** — Character normalization → Tokenization → POS-guided WordNetLemmatization with custom negation-preserving stopword list
2. **TF-IDF Vectorization** — Converts unstructured text to CSR sparse mathematical matrices, saving ~99% memory vs dense arrays
3. **Multinomial Naive Bayes Classifier** — Trained on 1,600 reviews, evaluated on 400 held-out reviews using Laplace smoothing
4. **Full Evaluation Suite** — Confusion matrix, word clouds (positive vs negative), top TF-IDF features per class, live inference function

### Key Engineering Insights Applied (from DecodeLabs Blueprint):
- Negations explicitly excluded from NLTK stopword list (default removes "not" — fatal for sentiment)
- POS tags passed to WordNetLemmatizer (without POS, verbs reduce incorrectly)
- N-grams (bigrams) capture negated phrases like "not good"
- CSR sparse format prevents memory crashes on high-dimensional text matrices
- Laplace smoothing prevents zero-frequency probability collapse on unseen words

---
*DecodeLabs Industrial Training Kit | Project 4: NLP & Sentiment Analysis | Batch 2026*  
*Anubhav Singh — GLA University AI/ML Bootcamp*
